## This script takes the Raw h5 files from Galdos et al, combines them together and does basic QC.

This data is of HIPSC cells differentiating/maturing into cardiomyocyte cells over 30 days

In [1]:
import anndata as ad
import numpy as np
import scanpy as sc
import scanpy.external as sce

Setting up some constants of file names and locations. HTO names are because this raw data doesn't actually set the day/cell type, and we have to do so ourselves with the given "antibody capture" data

In [2]:
prefix = "../rawFiles/Galdos/"
suffix = "feature_bc_matrix.h5"
Galdos_Hipsc = [
        prefix + "GSM6118768_Galdos_Seq_Run1_filtered_" + suffix,
        prefix + "GSM6118769_Galdos_Seq_Run2_filtered_" + suffix,
        prefix + "GSM6118770_Galdos_Seq_Run3_filtered_" + suffix,
    ]

HTOnames = [['WTC_DAY1', 'WTC_DAY2', 'WTC_DAY3', 'WTC_DAY4', 'WTC_DAY5', 'WTC_DAY6', 'LMNA_DAY1', 'LMNA_DAY2', 'LMNA_DAY3', 'LMNA_DAY4', 'LMNA_DAY5', 'LMNA_DAY6'],
["WTC_DAY0", "WTC_DAY7", "WTC_DAY11", "LMNA_DAY0", "LMNA_DAY7", "LMNA_DAY11"],
["WTC_DAY13", "WTC_DAY15", "WTC_DAY30", "LMNA_DAY13", "LMNA_DAY15", "LMNA_DAY30"]
]

Loop for getting all the data from the raw files and processing it. Processing them individually before joining, as there is a very strong chance it will crash If I try to join all 3 large unfiltered sets

In [ ]:
annot = sc.queries.biomart_annotations(
    "hsapiens",
    ["ensembl_gene_id", "external_gene_name", "description"],
    host="www.ensembl.org"
)
annot = annot.drop_duplicates(subset=['external_gene_name'], keep='first')
symbol_to_ensembl = annot.set_index("external_gene_name")["ensembl_gene_id"]

In [4]:
GaldosAdatas = []
for i, filePath in enumerate(Galdos_Hipsc):
    print(filePath)
    # get both Gene Expression (gex) and HTO data
    sample_adata = sc.read_10x_h5(filePath, gex_only=False)
    sample_adata.var_names_make_unique()

    # Split into Gene Expression (GEX) and HTO/Antibody capture
    is_hto = sample_adata.var["feature_types"] == "Antibody Capture"
    gex_adata = sample_adata[:, ~is_hto].copy()
    hto_adata = sample_adata[:, is_hto].copy()
    del sample_adata
    
    # Move HTO counts to obs in the gex_adata
    for j, hto_name in enumerate(hto_adata.var_names):
        gex_adata.obs[hto_name] = hto_adata.X[:, j].toarray().flatten()
    sce.pp.hashsolo(gex_adata, list(HTOnames[i]))
    gex_adata = gex_adata[~gex_adata.obs['Classification'].isin(["Doublet", "Negative", "None"]), :].copy()
    
    # mitochondrial genes
    gex_adata.var["mt"] = gex_adata.var_names.str.startswith("MT-")
    # ribosomal genes
    gex_adata.var["ribo"] = gex_adata.var_names.str.startswith(("RPS", "RPL"))
    # hemoglobin genes.
    gex_adata.var["hb"] = gex_adata.var_names.str.contains(r"^HB[ABDEGMQZ]\d*(?!\w)")
    sc.pp.calculate_qc_metrics(
        gex_adata, qc_vars=["mt", "ribo", "hb"], inplace=True, percent_top=[20], log1p=True
    )
    gex_adata = gex_adata[gex_adata.obs["pct_counts_mt"] < 20]

    gex_adata.obs["batch"] = "run_" + str(i)
    gex_adata.obs["batch"] = gex_adata.obs["batch"].astype("category")

    gex_adata.var['ensembl'] = gex_adata.var.index.map(symbol_to_ensembl)
    gex_adata.var['geneID'] = gex_adata.var_names
    print(gex_adata.var['geneID'])
    gex_adata.var_names = gex_adata.var['ensembl']
    gex_adata = gex_adata[:, gex_adata.var['ensembl'].notna()].copy()
    gex_adata.var_names_make_unique()

    gex_adata.layers["counts"] = gex_adata.X.copy()
    sc.pp.normalize_total(gex_adata, target_sum=1e4)
    sc.pp.log1p(gex_adata)

    GaldosAdatas.append(gex_adata)


../rawFiles/Galdos/GSM6118768_Galdos_Seq_Run1_filtered_feature_bc_matrix.h5


/home/kai/Documents/SCTLM-Bridges/.venv/lib/python3.14/site-packages/anndata/_core/anndata.py:1884: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


Please cite HashSolo paper:
https://www.cell.com/cell-systems/fulltext/S2405-4712(20)30195-2


<positron-console-cell-4>:31: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.


MIR1302-2HG    MIR1302-2HG
FAM138A            FAM138A
OR4F5                OR4F5
AL627309.1      AL627309.1
AL627309.3      AL627309.3
                  ...     
AC141272.1      AC141272.1
AC023491.2      AC023491.2
AC007325.1      AC007325.1
AC007325.4      AC007325.4
AC007325.2      AC007325.2
Name: geneID, Length: 36601, dtype: object
../rawFiles/Galdos/GSM6118769_Galdos_Seq_Run2_filtered_feature_bc_matrix.h5


...

MIR1302-2HG    MIR1302-2HG
FAM138A            FAM138A
OR4F5                OR4F5
AL627309.1      AL627309.1
AL627309.3      AL627309.3
                  ...     
AC141272.1      AC141272.1
AC023491.2      AC023491.2
AC007325.1      AC007325.1
AC007325.4      AC007325.4
AC007325.2      AC007325.2
Name: geneID, Length: 36601, dtype: object
../rawFiles/Galdos/GSM6118770_Galdos_Seq_Run3_filtered_feature_bc_matrix.h5


/home/kai/Documents/SCTLM-Bridges/.venv/lib/python3.14/site-packages/anndata/_core/anndata.py:1884: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


Please cite HashSolo paper:
https://www.cell.com/cell-systems/fulltext/S2405-4712(20)30195-2


<positron-console-cell-4>:31: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.


MIR1302-2HG    MIR1302-2HG
FAM138A            FAM138A
OR4F5                OR4F5
AL627309.1      AL627309.1
AL627309.3      AL627309.3
                  ...     
AC141272.1      AC141272.1
AC023491.2      AC023491.2
AC007325.1      AC007325.1
AC007325.4      AC007325.4
AC007325.2      AC007325.2
Name: geneID, Length: 36601, dtype: object


In [5]:
GaldosAdatas = ad.concat(GaldosAdatas, join="outer")
GaldosAdatas.var_names_make_unique()
GaldosAdatas.obs_names_make_unique()
sc.pp.filter_cells(GaldosAdatas, min_genes=200)
sc.pp.filter_genes(GaldosAdatas, min_cells=3)

/home/kai/Documents/SCTLM-Bridges/.venv/lib/python3.14/site-packages/anndata/_core/anndata.py:1882: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


Some object variable naming, a bit of a mess and I've added a bunch recently.

In [6]:
GaldosAdatas.obs["CellLine"] = GaldosAdatas.obs["Classification"].str.extract(r"^([^_]+)")
GaldosAdatas.obs["Day"] = (
    "DAY_" + GaldosAdatas.obs["Classification"].str.extract(r"DAY(\d+)")
    .iloc[:, 0]  # str.extract returns a DataFrame, pull the first column
    .str.zfill(2)
).astype("category")
toRemove = GaldosAdatas.obs.columns[GaldosAdatas.obs.columns.str.contains(r"_DAY")].tolist()
GaldosAdatas.obs.drop(columns = toRemove, inplace = True)

GaldosAdatas.obs["source"] = "Galdos"

GaldosAdatas.obs["substrate"] = "2D"
GaldosAdatas.obs["system"] = "hipsc"
GaldosAdatas.obs.rename(columns = {"Classification" : "condition"}, inplace = True)



Save the file for latter processing

In [7]:
GaldosAdatas.write_zarr("ProcessedData/GaldosScanpy.zarr")

/home/kai/Documents/SCTLM-Bridges/.venv/lib/python3.14/site-packages/anndata/_io/zarr.py:44: UserWarning: Writing zarr v2 data will no longer be the default in the next minor release. v3 data will be written by default. If you are explicitly setting this configuration, consider migrating to the zarr v3 file format.
  f = open_write_group(store)


If you need to get the data without rerunning everything again, this is here

In [8]:
GaldosAdatas = ad.read_zarr("ProcessedData/GaldosScanpy.zarr")